# Keyboard Hints Modal

> Modal-based keyboard shortcut reference with scannable grouped layout and `?` key trigger.

In [ ]:
#| default_exp components.hints_modal

In [ ]:
#| export
from __future__ import annotations
from typing import Optional, Sequence
from fasthtml.common import Div, Span, Dialog, Button, Form, H3, Kbd, Script, FT

from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.components.hints import (
    derive_navigation_hints,
    group_actions_by_zone_and_hint_group,
    mode_context_label,
)

from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_backdrop
from cjm_fasthtml_daisyui.components.actions.button import btn_modifiers
from cjm_fasthtml_daisyui.components.data_display.kbd import kbd as kbd_cls, kbd_sizes as kbd_sz
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles, badge_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, border_dui

from cjm_fasthtml_design_system.text_tiers import text_tiers

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, items, gap, justify, flex_direction,
)
from cjm_fasthtml_tailwind.utilities.layout import position, right, top, columns, break_util
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V1 button roles, V11 icon-size roles)
from cjm_fasthtml_design_system.buttons import buttons
from cjm_fasthtml_design_system.icons import icons, IconSize

## Key Display

In [ ]:
#| export
def _render_key_combo(
    display_key: str,  # formatted key combo string (e.g., "Ctrl+Shift+\u2191")
) -> Div:              # container with kbd elements for each key part
    """Render a key combination as a sequence of kbd elements joined by `+`."""
    parts = display_key.split('+')
    elements = []
    for i, part in enumerate(parts):
        if i > 0:
            elements.append(Span('+', cls=combine_classes(text_tiers.muted, m.x(0.5))))
        elements.append(Kbd(part.strip(), cls=combine_classes(kbd_cls, kbd_sz.sm)))
    return Div(*elements, cls=combine_classes(flex_display, items.center))

In [ ]:
from fasthtml.common import to_xml

# Single key
html = to_xml(_render_key_combo("Space"))
assert "kbd" in html
assert "Space" in html
assert "+" not in html.split("kbd")[0]  # no plus before first kbd

# Multi-key combo
html = to_xml(_render_key_combo("Ctrl+Shift+\u2191"))
assert html.count("kbd") >= 3  # 3 kbd elements (tag appears in open+close)
assert "Ctrl" in html
assert "Shift" in html
assert "\u2191" in html
print("Key display tests passed")

Key display tests passed


## Hint Row & Group

In [ ]:
#| export
def _render_hint_row(
    display_key: str,                       # formatted key combo string
    description: str,                       # action description
    mode_label: Optional[str] = None,        # mode-context chip text (e.g., "split", "default"); None for no chip
) -> Div:                                    # single shortcut row with key, description, and optional mode chip
    """Render a single shortcut row: key combo on left, description (with optional mode chip) on right."""
    description_components = [Span(description, cls=combine_classes(text_tiers.secondary))]
    if mode_label is not None:
        description_components.append(
            Span(
                mode_label,
                cls=combine_classes(
                    badge, badge_styles.soft, badge_sizes.xs,
                    m.l(2),
                ),
            )
        )
    return Div(
        _render_key_combo(display_key),
        Div(
            *description_components,
            cls=combine_classes(flex_display, items.center),
        ),
        cls=combine_classes(
            flex_display, items.center, justify.between,
            gap(4), p.y(1),
        )
    )


def _render_modal_group(
    group_name: str,                                                   # group header text
    rows: list[tuple[str, str, Optional[str]]],                         # list of (display_key, description, mode_label)
) -> Div:                                                              # group container with header and rows
    """Render a group of related shortcuts with a header.

    `break_util.inside.avoid_column` keeps the entire group (header + rows)
    in one column inside the modal body's CSS-columns layout — groups never
    split across columns, preserving "form follows function" scannability.
    """
    row_elements = [_render_hint_row(key, desc, mode) for key, desc, mode in rows]
    return Div(
        Div(
            group_name,
            cls=combine_classes(
                font_size.xs, font_weight.semibold,
                text_tiers.muted,
                p.b(1),
                border.b(), border_dui.base_content.opacity(10),
                m.b(1),
            )
        ),
        *row_elements,
        cls=combine_classes(m.b(4), break_util.inside.avoid_column),
    )

In [ ]:
# Test hint row (without mode chip)
row_html = to_xml(_render_hint_row("Ctrl+Z", "Undo last action"))
assert "Ctrl" in row_html
assert "Z" in row_html
assert "Undo last action" in row_html
# No mode chip when mode_label is omitted (defaults to None)
assert "badge-soft" not in row_html, "No mode chip should render when mode_label is None"

# Test hint row WITH mode chip
chip_row_html = to_xml(_render_hint_row("Enter", "Split at caret", mode_label="split"))
assert "Split at caret" in chip_row_html
assert ">split<" in chip_row_html, "Mode chip text must appear in the rendered row"
assert "badge-soft" in chip_row_html, "Mode chip must use the daisyui soft badge style"
assert "badge-xs" in chip_row_html, "Mode chip must use the xs badge size"

# Test modal group (new 3-tuple signature: (key, desc, mode_label))
group_html = to_xml(_render_modal_group("Editing", [
    ("Enter", "Enter split mode", None),
    ("Escape", "Exit split mode", "split"),  # mode-restricted row gets a chip
]))
assert "Editing" in group_html
assert "Enter split mode" in group_html
assert "Exit split mode" in group_html
assert ">split<" in group_html  # mode chip rendered on second row only

# Group container must carry break-inside-avoid-column for CSS-columns layout
assert "break-inside-avoid-column" in group_html, \
    "Modal group container must use break-inside-avoid-column to stay intact across columns"

print("Hint row and group tests passed")

## Modal Body

In [ ]:
#| export
def _render_section_header(
    label: str,  # human-readable section label
) -> Div:        # styled section-header element
    """Render a section header for one manager in a multi-manager modal.

    Visually distinct from group headers (`_render_modal_group`):
    - Section headers: font-sm + bold + secondary tier + top border + extra spacing
    - Group headers:   font-xs + semibold + muted tier + bottom border (no top)

    `break_util.after.avoid` keeps a section header attached to its first
    group inside the CSS-columns layout — prevents the orphan-header case where
    a header lands at the bottom of one column and its first group at the top
    of the next. (`break_util.inside.avoid_column` exists but only applies to
    breaks INSIDE the element; for "don't break right after", `break-after: avoid`
    is the correct CSS property.)
    """
    return Div(
        label,
        cls=combine_classes(
            font_size.sm, font_weight.bold,
            text_tiers.secondary,
            p.t(3), p.b(2),
            m.t(2), m.b(2),
            border.t(2), border_dui.base_content.opacity(20),
            break_util.after.avoid,
        ),
    )


# Built-in group label that the manager-derived Navigation rows render under.
# Consumer actions with `hint_group=_BUILTIN_NAV_GROUP` in the shared section
# get merged into the same group (no duplicate header).
_BUILTIN_NAV_GROUP = "Navigation"


def _render_manager_groups(
    manager: ZoneManager,                # the manager to render groups for
    include_zone_switch: bool = True,    # include zone-switch hint when multi-zone
) -> list[FT]:                           # flat list of group FT elements (no wrapping Div)
    """Render the keyboard-shortcut groups for a single ZoneManager.

    Composes:
    1. **Manager-derived "Navigation" group**: derived nav rows from
       `derive_navigation_hints(manager)` (via `manager.key_mapping` + each zone's
       `navigation.get_supported_directions()`), plus the Switch-panel row when
       `include_zone_switch=True` and multi-zone. **Plus** any consumer actions
       whose `hint_group == "Navigation"` AND land in the shared section
       (zone_label=None) — they fold into the same group rather than rendering
       a duplicate "Navigation" header underneath the derived rows.
    2. **Zone-aware action groups**: from `group_actions_by_zone_and_hint_group`,
       with `"<zone label> — <hint_group>"` headers for per-zone sections and
       plain `"<hint_group>"` for shared sections (other than Navigation).

    Per-zone "Navigation" groups stay separate (they're zone-scoped — distinct
    from the manager-level Navigation row). Only the shared-section Navigation
    rows merge into the manager-derived group.

    Returned as a flat list (no wrapping Div) so callers can interleave section
    headers between groups when rendering multi-manager hierarchies.
    """
    from cjm_fasthtml_keyboard_navigation.core.key_mapping import format_key_for_display

    elements: list[FT] = []

    # Build the manager-derived "Navigation" group's rows
    nav_rows: list[tuple[str, str, Optional[str]]] = [
        (display_key, description, None)
        for display_key, description in derive_navigation_hints(manager)
    ]
    if include_zone_switch and len(manager.zones) > 1:
        prev_key = format_key_for_display(manager.prev_zone_key)
        next_key = format_key_for_display(manager.next_zone_key)
        nav_rows.append((f"{prev_key} / {next_key}", "Switch panel", None))

    # Walk action groups, merging shared-section "Navigation" into nav_rows
    # and collecting the rest for downstream emission as their own groups.
    remaining_groups: list[tuple[Optional[str], str, list[KeyAction]]] = []
    for zone_label, group_name, actions in group_actions_by_zone_and_hint_group(manager):
        if zone_label is None and group_name == _BUILTIN_NAV_GROUP:
            # Shared-section Navigation actions: fold into the manager-derived
            # Navigation group so the modal shows one "Navigation" header, not two.
            nav_rows.extend(
                (a.get_display_key(), a.description, mode_context_label(a))
                for a in actions
            )
        else:
            remaining_groups.append((zone_label, group_name, actions))

    # Emit the merged Navigation group (if any rows were collected)
    if nav_rows:
        elements.append(_render_modal_group(_BUILTIN_NAV_GROUP, nav_rows))

    # Emit remaining groups (per-zone, or non-Navigation shared)
    for zone_label, group_name, actions in remaining_groups:
        rows = [
            (a.get_display_key(), a.description, mode_context_label(a))
            for a in actions
        ]
        header = f"{zone_label} — {group_name}" if zone_label else group_name
        elements.append(_render_modal_group(header, rows))

    return elements


def _render_modal_body(
    manager: ZoneManager,                            # primary keyboard zone manager
    include_zone_switch: bool = True,                # include zone-switch hint (single-manager: when multi-zone; multi-manager: per-manager)
    include_navigation: bool = True,                 # DEPRECATED: no-op. Built-in nav row derived from key_mapping now.
    child_managers: Optional[Sequence[ZoneManager]] = None,  # additional managers for hierarchical hint display (each renders as a labeled section)
) -> Div:                                            # modal body with grouped shortcuts
    """Render the modal body with grouped keyboard shortcuts.

    **Single-manager mode** (`child_managers=None`, the common case):
    - Renders one manager's content as a flat list of groups (no section header).
    - Layout: derived "Navigation" group → zone-aware action groups.

    **Multi-manager mode** (`child_managers=[...]`, hierarchical keyboard systems):
    - Renders the primary manager as a labeled section (header = `manager.get_display_label()`),
      followed by each child manager as its own labeled section. Used for
      coordinator-based hierarchies where multiple ZoneManagers cooperate
      (e.g., parent + N children pattern in `cjm-fasthtml-keyboard-navigation`'s
      hierarchy demo). Each manager's content composes via `_render_manager_groups`.

    Layout details (same in both modes):
    - Each group container has `break-inside: avoid-column` so the CSS-columns
      layout never splits a group across columns.
    - Section headers (multi-manager mode) carry `break-after: avoid`
      to keep each header attached to its first group.
    - Mode-restricted actions get a small chip ("split", "default", etc.)
      derived from `mode_context_label(action)`.
    - Per-manager: shared-section actions with `hint_group="Navigation"` fold
      into the manager-derived Navigation group (no duplicate group headers).

    History: an earlier version hardcoded `↑/↓ Navigate items` regardless of
    the manager's actual `key_mapping`. That row was wrong under custom
    mappings (wasd, vim) and was dropped in G4. This version derives the
    nav row from `manager.key_mapping` via `derive_navigation_hints`, which
    is accurate under any KeyMapping configuration.
    """
    elements: list[FT] = []

    if child_managers:
        # Multi-manager mode: label the primary manager too, so the hierarchy
        # is visually clear in the rendered modal.
        elements.append(_render_section_header(manager.get_display_label()))

    elements.extend(_render_manager_groups(manager, include_zone_switch=include_zone_switch))

    if child_managers:
        for child in child_managers:
            elements.append(_render_section_header(child.get_display_label()))
            elements.extend(_render_manager_groups(child, include_zone_switch=include_zone_switch))

    # `columns.sm` = CSS column-width: 24rem; browser auto-decides count based
    # on modal's actual rendered width. Short content stays effectively single-
    # column at narrow modal widths; long content (e.g., multi-manager modals)
    # auto-flows into 2-3 columns at wider modal sizes.
    return Div(
        *elements,
        cls=combine_classes(p.t(2), columns.sm),
    )

In [ ]:
from cjm_fasthtml_keyboard_navigation.core.focus_zone import FocusZone
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.core.key_mapping import WASD_KEYS
from cjm_fasthtml_keyboard_navigation.core.navigation import LinearVertical

# --- test_manager: zones WITHOUT item_selector ---
# derive_navigation_hints returns [] when no zone has items, so this fixture
# tests the consumer-action path without a manager-derived nav row.
z1 = FocusZone(id="seg")
z2 = FocusZone(id="align")
test_manager = ZoneManager(
    zones=(z1, z2),
    actions=(
        KeyAction(key="Enter", js_callback="x", description="Enter split mode", hint_group="Editing"),
        KeyAction(key="Escape", js_callback="x", description="Exit split mode", hint_group="Editing"),
        KeyAction(key="Backspace", htmx_trigger="x", description="Merge with previous", hint_group="Editing"),
        KeyAction(key="z", modifiers=frozenset({"ctrl"}), htmx_trigger="x", description="Undo", hint_group="Editing"),
        KeyAction(key=" ", js_callback="x", description="Play audio", hint_group="Audio"),
    ),
    prev_zone_key="ArrowLeft",
    next_zone_key="ArrowRight",
)

body_html = to_xml(_render_modal_body(test_manager))
# Manager-derived Navigation group (Switch panel only — no item_selector zones,
# so derive_navigation_hints returns empty)
assert "Navigation" in body_html
assert "Switch panel" in body_html
# Consumer action groups (zone_ids=None → all go to shared section)
assert "Editing" in body_html
assert "Enter split mode" in body_html
assert "Audio" in body_html
assert "Play audio" in body_html
# No derived nav rows: test_manager's zones have no item_selector
assert "Navigate items" not in body_html, \
    "Without item_selector zones, derive_navigation_hints returns empty — no nav rows expected"

print("Modal body tests passed (no-item-selector manager)")


# --- Test: derived nav row WITH item_selector zones (ARROW_KEYS default) ---
nav_z1 = FocusZone(id="nav-seg", item_selector="li", navigation=LinearVertical())
nav_z2 = FocusZone(id="nav-align", item_selector="li", navigation=LinearVertical())
nav_manager = ZoneManager(
    zones=(nav_z1, nav_z2),
    actions=(),
    prev_zone_key="ArrowLeft",
    next_zone_key="ArrowRight",
)
nav_body_html = to_xml(_render_modal_body(nav_manager))
# Derived "↑ / ↓ Navigate items" row appears (default ARROW_KEYS + LinearVertical).
# _render_key_combo renders the combo as a single Kbd containing "↑ / ↓"
# (only splits on '+'), so check the literal combo string in the HTML.
assert "Navigate items" in nav_body_html
assert "↑ / ↓" in nav_body_html, "ARROW_KEYS LinearVertical must surface as '↑ / ↓'"
# Switch-panel row also appears (multi-zone)
assert "Switch panel" in nav_body_html


# --- Test: WASD regression fix — modal shows W/S, NOT ↑/↓ ---
# This is the load-bearing assertion for the WASD bug fix. Pre-fix, the modal
# either hardcoded ↑/↓ (lie) or showed nothing (regression after the drop).
# Post-fix, the modal shows the actually-bound keys via derive_navigation_hints.
wasd_z = FocusZone(id="wasd-list", item_selector="li", navigation=LinearVertical())
wasd_manager = ZoneManager(zones=(wasd_z,), key_mapping=WASD_KEYS)
wasd_body_html = to_xml(_render_modal_body(wasd_manager))
assert "w / s" in wasd_body_html, \
    "WASD-mapped manager must show 'w / s' as the navigation keys in the modal"
assert "↑" not in wasd_body_html and "↓" not in wasd_body_html, \
    "WASD-mapped manager must NOT show arrow-key symbols — that was the latent bug"

print("Derived navigation hints in modal body tests passed")

## Trigger Button

In [ ]:
#| export
def render_keyboard_hints_trigger(
    modal_id: str = "kb-hints-modal",             # ID of the modal dialog to open
    icon_size: IconSize = icons.ghost_button,     # lucide icon size (V11.R3 ghost-button: "full" — pairs with V1.modal_disclosure at btn-xs)
) -> Button:                                      # ghost button with keyboard icon
    """Render a keyboard icon button that opens the hints modal."""
    return Button(
        lucide_icon("keyboard", size=icon_size),
        cls=combine_classes(buttons.modal_disclosure, btn_modifiers.circle),
        title="Keyboard shortcuts (?)",
        onclick=f"document.getElementById('{modal_id}').showModal();",
        type="button",
    )

In [ ]:
trigger = render_keyboard_hints_trigger()
html = to_xml(trigger)
assert "keyboard" in html.lower() or "svg" in html  # has icon
assert "showModal" in html
assert "kb-hints-modal" in html
assert 'title="Keyboard shortcuts (?)"' in html
print("Trigger button tests passed")

Trigger button tests passed


## Question Mark Key Listener

In [ ]:
#| export
def _render_question_mark_listener(
    modal_id: str,  # ID of the modal dialog to toggle
) -> Script:        # script element with global `?` key listener
    """Render a global `?` key listener that toggles the hints modal.
    
    Uses a named function stored on `window` so that HTMX re-renders
    replace the previous listener instead of accumulating duplicates.
    """
    return Script(f"""
    (function() {{
        // Remove previous listener if it exists (HTMX re-render dedup)
        if (window._kbHintsKeyListener) {{
            document.removeEventListener('keydown', window._kbHintsKeyListener);
        }}
        window._kbHintsKeyListener = function(e) {{
            // Skip if typing in an input, textarea, or contenteditable
            var tag = e.target.tagName;
            if (tag === 'INPUT' || tag === 'TEXTAREA' || e.target.isContentEditable) return;
            if (e.key === '?') {{
                e.preventDefault();
                var m = document.getElementById('{modal_id}');
                if (m) {{
                    if (m.open) {{ m.close(); }}
                    else {{ m.showModal(); }}
                }}
            }}
        }};
        document.addEventListener('keydown', window._kbHintsKeyListener);
    }})();
    """)


In [ ]:
listener = _render_question_mark_listener("kb-hints-modal")
html = to_xml(listener)
assert "keydown" in html
assert "e.key === '?'" in html
assert "showModal" in html
assert "INPUT" in html  # skips input fields
assert "TEXTAREA" in html
assert "isContentEditable" in html
print("Question mark listener tests passed")

Question mark listener tests passed


## Full Modal Component

In [ ]:
#| export
def render_keyboard_hints_modal(
    manager: ZoneManager,                                   # primary keyboard zone manager
    modal_id: str = "kb-hints-modal",                        # HTML ID for the modal dialog
    include_navigation: bool = True,                         # DEPRECATED: no-op kept for backward compat. See _render_modal_body.
    include_zone_switch: bool = True,                        # include zone-switch hint (auto-hidden for single zone)
    enable_question_mark_key: bool = True,                   # add global `?` key listener
    title: str = "Keyboard Shortcuts",                       # modal title text
    child_managers: Optional[Sequence[ZoneManager]] = None,  # child managers for hierarchical hint display (each rendered as a labeled section)
) -> tuple[FT, FT, FT]:                                     # (modal_dialog, trigger_button, question_mark_script)
    """Render a modal-based keyboard shortcut reference.

    Returns three components:
    - `modal_dialog`: The Dialog element (place anywhere in page)
    - `trigger_button`: Small keyboard icon button (place in step header)
    - `question_mark_script`: Global `?` key listener Script (place in page)

    If `enable_question_mark_key` is False, `question_mark_script` is an empty Div.

    **Hierarchical hints** (`child_managers=[...]`): when working with multiple
    ZoneManagers coordinated by `window.kbCoordinator` (parent + N children), pass
    the parent as `manager` and the children as `child_managers`. The modal will
    render each as a labeled section using `manager.get_display_label()` for
    section headers. Set `label` on each ZoneManager for human-readable headers;
    falls back to `system_id` otherwise.

    Modal width ladder (R2 cap + optimal-space response — see
    layout-system.md M1–M6 modes): grows responsively with viewport. Combined
    with `columns.sm` on the body, this gives 1 column at narrow widths,
    2 columns at laptop full-screen (lg breakpoint with max_w._4xl), and
    3 columns at desktop full-screen (2xl breakpoint with max_w._7xl).
    Modal width is an upper bound; DaisyUI's `modal_box` sizes the actual
    modal to its content within that bound, so short content stays compact.
    """
    body = _render_modal_body(
        manager,
        include_zone_switch=include_zone_switch,
        child_managers=child_managers,
    )

    modal_dialog = Dialog(
        Div(
            # Close button (top-right corner)
            Form(
                Button(
                    "✕",
                    cls=combine_classes(
                        buttons.soft_dismissal, btn_modifiers.circle,
                        position.absolute, right._2, top._2,
                    ),
                ),
                method="dialog",
            ),
            # Title
            H3(
                lucide_icon("keyboard", size=icons.section_header, cls=str(m.r(2))),
                title,
                cls=combine_classes(
                    font_size.lg, font_weight.bold,
                    flex_display, items.center,
                ),
            ),
            # Shortcut groups
            body,
            # Footer hint
            Div(
                Span("Press "),
                Kbd("?", cls=combine_classes(kbd_cls, kbd_sz.sm)),
                Span(" to toggle this dialog"),
                cls=combine_classes(
                    font_size.xs, text_tiers.subtle,
                    p.t(3), border.t(), border_dui.base_content.opacity(10),
                    flex_display, items.center, gap(1),
                ),
            ),
            cls=combine_classes(
                modal_box,
                # Responsive max-width ladder per layout-system M1–M6 modes.
                # Values tuned via cross-device retest 2026-05-11: laptop
                # full-screen wants 2-col (lg + max_w._4xl); desktop full-screen
                # wants 3-col (2xl + max_w._7xl). Content-driven actual width:
                # modal shrinks below the cap when content is short.
                max_w.md,           # M1 Pocket base
                max_w.lg.sm,        # M2 Compact (sm 640+)
                max_w._4xl.lg,      # M3 Standard (lg 1024+) — 2-col on laptop
                max_w._7xl._2xl,    # M4+ Spacious (2xl 1536+) — 3-col on desktop
            ),
        ),
        # Backdrop (click outside to close)
        Form(Button("close"), method="dialog", cls=str(modal_backdrop)),
        id=modal_id,
        cls=str(modal),
    )

    trigger = render_keyboard_hints_trigger(modal_id=modal_id)

    question_mark_script = (
        _render_question_mark_listener(modal_id)
        if enable_question_mark_key
        else Div(style="display:none;")
    )

    return modal_dialog, trigger, question_mark_script

In [ ]:
# Test dual-zone modal: zone-aware sectioning + dropped hardcoded nav row + mode chips
modal_dialog, trigger, qm_script = render_keyboard_hints_modal(test_manager)

modal_html = to_xml(modal_dialog)
trigger_html = to_xml(trigger)
script_html = to_xml(qm_script)

# Modal structure
assert 'id="kb-hints-modal"' in modal_html
assert 'modal-box' in modal_html
assert 'modal-backdrop' in modal_html
assert 'Keyboard Shortcuts' in modal_html
assert 'Navigation' in modal_html  # manager-derived Switch-panel group label
assert 'Editing' in modal_html
assert 'Audio' in modal_html
assert '✕' in modal_html  # close button

# Note: test_manager's zones have no item_selector, so derive_navigation_hints
# returns []. The modal's "Navigation" group renders only the Switch-panel row.
# (Tests for the derived nav row WITH item_selectors live in the modal-body
# test cell above — including the WASD regression guard.)
assert 'Navigate items' not in modal_html, \
    "test_manager has no item_selectors → no derived nav rows expected"

# The Switch-panel hint IS still emitted (key-mapping-derived, correct under custom mappings)
assert 'Switch panel' in modal_html

# Optimal-space layout: CSS columns + responsive max_w + break-inside guard.
# Width-ladder values tuned via cross-device retest 2026-05-11 — laptop wants
# 2-col at lg with max_w._4xl, desktop wants 3-col at 2xl with max_w._7xl.
assert 'columns-sm' in modal_html, "Modal body must use columns-sm for CSS auto-distribution"
assert 'break-inside-avoid-column' in modal_html, "Groups must use break-inside-avoid-column"
assert 'max-w-md' in modal_html        # M1 base
assert 'sm:max-w-lg' in modal_html      # M2 step
assert 'lg:max-w-4xl' in modal_html     # M3 step — 2-col on laptop
assert '2xl:max-w-7xl' in modal_html    # M4+ step — 3-col on desktop

# Trigger
assert 'showModal' in trigger_html
assert 'kb-hints-modal' in trigger_html

# Question mark listener
assert 'keydown' in script_html
assert "e.key === '?'" in script_html

# Test with question mark key disabled
_, _, no_qm = render_keyboard_hints_modal(test_manager, enable_question_mark_key=False)
no_qm_html = to_xml(no_qm)
assert 'keydown' not in no_qm_html  # no listener
assert 'display:none' in no_qm_html  # empty placeholder

# Test single-zone manager (no zone switch hint, no zone-label prefix)
single_manager = ZoneManager(
    zones=(z1,),
    actions=(KeyAction(key=" ", js_callback="x", description="Select", hint_group="Actions"),),
)
single_modal, _, _ = render_keyboard_hints_modal(single_manager)
single_html = to_xml(single_modal)
assert 'Switch panel' not in single_html  # no zone switch for single zone
assert 'Select' in single_html  # consumer action surfaces
# Single-zone managers should NOT emit "seg — Actions" style headers
assert ' — ' not in single_html, "Single-zone manager must not emit zone-label-prefixed group headers"
print("Full modal component tests passed")

# --- G4 regression-guard pattern: dual-zone shared-factory ---
# Reproduces segment-align's exact shape. Without zone-scoping in the grouper,
# rows like "Previous item" would duplicate inside a single "Navigation" header.
zone_seg = FocusZone(id="seg", label="Text Segmentation")
zone_align = FocusZone(id="align", label="VAD Alignment")

def _shared_factory_g4(zid):
    return (
        KeyAction(key="ArrowUp", htmx_trigger=f"{zid}-up", zone_ids=(zid,),
                  description="Previous item", hint_group="Navigation"),
        KeyAction(key="ArrowDown", htmx_trigger=f"{zid}-down", zone_ids=(zid,),
                  description="Next item", hint_group="Navigation"),
    )

g4_manager = ZoneManager(
    zones=(zone_seg, zone_align),
    actions=(
        *_shared_factory_g4("seg"),
        *_shared_factory_g4("align"),
        # Mode-restricted action — must render with a mode chip
        KeyAction(key="Enter", htmx_trigger="x", zone_ids=("seg",),
                  mode_names=("split",),
                  description="Split at caret", hint_group="Split Mode"),
        # not_modes action — must render with "default" chip
        KeyAction(key="Backspace", htmx_trigger="m", zone_ids=("seg",),
                  not_modes=("split",),
                  description="Merge with previous", hint_group="Segmentation"),
    ),
)

g4_modal, _, _ = render_keyboard_hints_modal(g4_manager)
g4_html = to_xml(g4_modal)

# Per-zone section headers: explicit zone-label prefix
assert "Text Segmentation — Navigation" in g4_html
assert "VAD Alignment — Navigation" in g4_html
assert "Text Segmentation — Split Mode" in g4_html
assert "Text Segmentation — Segmentation" in g4_html

# Mode chip presence on mode-restricted rows
assert ">split<" in g4_html, "Mode chip 'split' must appear on mode_names=('split',) action"
assert ">default<" in g4_html, "Mode chip 'default' must appear on not_modes=('split',) action"

# Hierarchy sentinel: Text Segmentation must appear BEFORE VAD Alignment
# in the rendered HTML (matches manager.zones declaration order). Prevents
# silent re-ordering by future renderer changes.
seg_first = g4_html.index("Text Segmentation — Navigation")
align_first = g4_html.index("VAD Alignment — Navigation")
assert seg_first < align_first, \
    "Per-zone sections must appear in zone declaration order (Text Segmentation before VAD Alignment)"

# Regression guard: "Previous item" must appear EXACTLY TWICE in the modal
# (once under "Text Segmentation — Navigation", once under "VAD Alignment —
# Navigation") — NOT 4 times (which would indicate the pre-fix duplication
# bug where both zones' actions collapsed into a single header).
assert g4_html.count(">Previous item<") == 2, \
    f"'Previous item' must appear exactly twice (once per zone), got {g4_html.count('>Previous item<')}"
assert g4_html.count(">Next item<") == 2, \
    f"'Next item' must appear exactly twice (once per zone), got {g4_html.count('>Next item<')}"

print("G4 regression-guard pattern tests passed")

In [ ]:
# Test multi-manager (hierarchical) hints modal
# Reproduces the hierarchy demo pattern: parent + two children, coordinated
# via window.kbCoordinator at runtime. The modal renders each as a labeled
# section using `manager.get_display_label()`.
from cjm_fasthtml_keyboard_navigation.core.navigation import LinearVertical, ScrollOnly

# Parent: ghost zones (ScrollOnly) for switching between areas.
# Includes a consumer action with hint_group="Navigation" — this exercises the
# Navigation-merge behavior: the consumer's Navigation action must fold into
# the manager-derived Navigation group (Switch panel), NOT produce a second
# "Navigation" header.
ghost_a = FocusZone(id="ghost-a", item_selector=None, navigation=ScrollOnly())
ghost_b = FocusZone(id="ghost-b", item_selector=None, navigation=ScrollOnly())
parent_mgr = ZoneManager(
    zones=(ghost_a, ghost_b),
    actions=(
        KeyAction(key="Enter", js_callback="activateChild",
                  description="Activate area", hint_group="Navigation"),
    ),
    label="Parent — Hierarchy Coordinator",
)

# Child A: LinearVertical list
child_a_zone = FocusZone(id="child-a-list", item_selector="li", navigation=LinearVertical())
child_a_mgr = ZoneManager(
    zones=(child_a_zone,),
    actions=(
        KeyAction(key=" ", htmx_trigger="child-a-toggle",
                  description="Toggle selection", hint_group="Selection"),
    ),
    label="Alpha List",
)

# Child B: LinearVertical list
child_b_zone = FocusZone(id="child-b-list", item_selector="li", navigation=LinearVertical())
child_b_mgr = ZoneManager(
    zones=(child_b_zone,),
    actions=(
        KeyAction(key=" ", htmx_trigger="child-b-toggle",
                  description="Toggle selection", hint_group="Selection"),
    ),
    label="Beta List",
)

# Render the hierarchical modal
hier_modal, _, _ = render_keyboard_hints_modal(
    parent_mgr,
    child_managers=(child_a_mgr, child_b_mgr),
)
hier_html = to_xml(hier_modal)

# Each manager's label appears as a section header
assert "Parent — Hierarchy Coordinator" in hier_html
assert "Alpha List" in hier_html
assert "Beta List" in hier_html

# Parent's content: Switch panel (multi-zone) + the Enter "Activate area" action
assert "Switch panel" in hier_html
assert "Activate area" in hier_html

# Children's content: their own derived nav rows ('↑ / ↓' for LinearVertical)
# AND their own action rows (Space — Toggle selection)
assert "↑ / ↓" in hier_html, "Child managers' LinearVertical pattern must surface ↑/↓"
# Toggle selection should appear TWICE (once per child)
assert hier_html.count(">Toggle selection<") == 2, \
    f"'Toggle selection' must appear twice (once per child), got {hier_html.count('>Toggle selection<')}"

# --- Regression guard: Navigation-merge behavior ---
# Per manager, the modal should emit exactly ONE "Navigation" group header
# (containing both manager-derived rows and consumer actions with
# hint_group="Navigation"). Across the 3 managers (parent + 2 children),
# we expect EXACTLY 3 "Navigation" group headers total — NOT 4 (which would
# indicate the parent's consumer Navigation action is producing its own
# duplicate header).
assert hier_html.count(">Navigation<") == 3, \
    f"Each manager must emit exactly one 'Navigation' header (3 total: parent + 2 children), got {hier_html.count('>Navigation<')}"

# Hierarchy sentinel: section headers appear in (parent → children) declaration order
parent_pos = hier_html.index("Parent — Hierarchy Coordinator")
alpha_pos = hier_html.index("Alpha List")
beta_pos = hier_html.index("Beta List")
assert parent_pos < alpha_pos < beta_pos, \
    "Section headers must appear in (parent, child_managers[0], child_managers[1], ...) order"

# Section headers must use break-after-avoid to stay attached to their groups
# (prevents orphan headers in CSS columns layout — `break-after-avoid` is the
# correct CSS for "don't break right after this element").
assert "break-after-avoid" in hier_html, \
    "Section headers must carry break-after-avoid to prevent orphan headers in CSS columns"


# --- Regression guard: single-manager call still works (no children) ---
# Without child_managers, no section headers should render (preserves the
# common-case modal layout).
single_modal, _, _ = render_keyboard_hints_modal(parent_mgr)
single_html = to_xml(single_modal)
assert "Alpha List" not in single_html  # children NOT rendered
assert "Beta List" not in single_html
# The parent's label SHOULDN'T appear either — single-manager mode is implicit context
assert "Parent — Hierarchy Coordinator" not in single_html, \
    "Single-manager mode must not emit a section header for the primary manager"
# break-after-avoid should NOT appear in single-manager output (no section headers)
assert "break-after-avoid" not in single_html, \
    "Single-manager mode must not render section headers (no break-after class)"
# Navigation-merge: still ONE "Navigation" header in single-manager mode too
assert single_html.count(">Navigation<") == 1, \
    f"Single-manager mode must emit exactly one 'Navigation' header, got {single_html.count('>Navigation<')}"


# --- Per-zone Navigation must STAY separate (zone-scoped vs shared) ---
# When a consumer registers actions with hint_group="Navigation" AND zone_ids
# tying them to specific zones, those zone-scoped Navigation groups must NOT
# merge into the manager-derived shared Navigation group. They render as
# their own "<zone label> — Navigation" headers.
z_seg = FocusZone(id="z-seg", label="Text Seg", item_selector="li", navigation=LinearVertical())
z_align = FocusZone(id="z-align", label="VAD", item_selector="li", navigation=LinearVertical())
zone_scoped_mgr = ZoneManager(
    zones=(z_seg, z_align),
    actions=(
        # Zone-scoped Navigation action — should land under "Text Seg — Navigation"
        # NOT merge into the manager-derived top-level Navigation group.
        KeyAction(key="ArrowUp", htmx_trigger="seg-prev", zone_ids=("z-seg",),
                  description="Previous item", hint_group="Navigation"),
    ),
)
zs_modal, _, _ = render_keyboard_hints_modal(zone_scoped_mgr)
zs_html = to_xml(zs_modal)
# Expect TWO Navigation headers: manager-derived top one + zone-scoped "Text Seg — Navigation"
assert ">Navigation<" in zs_html  # manager-derived
assert "Text Seg — Navigation" in zs_html  # zone-scoped, must stay separate


# --- Label fallback: child without explicit label uses system_id ---
unlabeled_child = ZoneManager(
    zones=(FocusZone(id="raw-zone"),),
    label=None,
)
fallback_modal, _, _ = render_keyboard_hints_modal(parent_mgr, child_managers=(unlabeled_child,))
fallback_html = to_xml(fallback_modal)
assert "raw-zone" in fallback_html, \
    "Child without explicit label must fall back to system_id (auto-derived from initial zone id)"

print("Hierarchical hints modal tests passed (incl. Navigation-merge regression guard)")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()